# 10 · 迁移学习与预训练模型

> 从零训练大模型需要海量数据和算力。迁移学习让你**站在巨人肩膀上**：
> 用 ImageNet 等大数据集训练好的权重，快速迁移到自己的任务。

## 10.1 为什么迁移学习有效

CNN 学到的特征具有**层次性**和**通用性**：
- **浅层**：边缘、颜色、纹理（几乎所有视觉任务通用）
- **中层**：图案、部件
- **深层**：语义对象（接近任务相关）

因此预训练模型的浅层几乎可以直接复用。

## 10.2 两种迁移策略

| 策略 | 做法 | 适用 |
|---|---|---|
| **特征提取** | 冻结主干，只训练新分类头 | 数据少、任务与预训练相近 |
| **微调 Fine-tune** | 解冻部分/全部层，小学习率训练 | 数据较多、任务差异大 |

经验法则：
- 数据量 < 1k：只训分类头
- 1k ~ 10k：训分类头 + 最后几个 block
- > 10k：全部微调（不同层可用不同 lr）

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torchvision:", torchvision.__version__)
print("device:", device)

## 10.3 加载预训练模型

`torchvision.models` 提供大量预训练模型。用 `weights="DEFAULT"`（新 API）
或 `pretrained=True`（旧 API）。

**首次运行会联网下载权重**（ResNet18 约 45MB）。无网络时会失败，下面做了兜底。

In [ ]:
def get_resnet18(pretrained=True, num_classes=1000):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    if num_classes != 1000:
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

try:
    model = get_resnet18(pretrained=True, num_classes=10)
    print("成功加载预训练权重")
except Exception as e:
    print("加载预训练失败（可能无网络）：", type(e).__name__)
    print("使用随机初始化继续演示")
    model = get_resnet18(pretrained=False, num_classes=10)

model = model.to(device)
print(model.fc)
print("总参数量:", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 查看网络结构，便于决定冻结哪些层
print(model)

## 10.4 策略一：特征提取（冻结主干）

**关键**：冻结层必须用 `p.requires_grad = False`，
否则优化器会更新它们。

In [ ]:
model = get_resnet18(pretrained=True, num_classes=10).to(device)

# 冻结所有参数
for p in model.parameters():
    p.requires_grad = False

# 替换分类头（新层默认 requires_grad=True）
model.fc = nn.Linear(model.fc.in_features, 10).to(device)

trainable = [p for p in model.parameters() if p.requires_grad]
print("可训练参数数:", sum(p.numel() for p in trainable))
print("全部参数数  :", sum(p.numel() for p in model.parameters()))

# 优化器只传可训练参数
optimizer = torch.optim.AdamW(trainable, lr=1e-3, weight_decay=1e-4)
print("优化器只优化分类头，参数量:", sum(p.numel() for p in optimizer.param_groups[0]["params"]))

### 小心 BatchNorm

即使冻结了参数，`BatchNorm` 在 **train 模式**下仍会更新 `running_mean/var`。
如果目标数据分布差异大，这会让统计量漂移。做法：

```python
model.train()
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.eval()          # 让 BN 保持预训练统计量
```

In [ ]:
def freeze_bn(model):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

# 训练时调用 freeze_bn(model) 即可（eval() 会覆盖 train()）
print("已定义 freeze_bn")

## 10.5 准备目标数据集

用 CIFAR-10 的子集模拟"小数据集"任务（取 4 类，每类 500 张），
输入 resize 到 224 适配 ImageNet 预训练模型。

> 真实项目中换成你自己的 `ImageFolder` 数据即可。

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ImageNet 预训练模型的标准化参数
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class RemapLabel(torch.utils.data.Dataset):
    def __init__(self, subset, mapping):
        self.subset = subset
        self.mapping = mapping
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, i):
        x, y = self.subset[i]
        return x, self.mapping[y]


def subset_by_classes(dataset, keep, per_class=None):
    targets = np.array(dataset.targets)
    idx = []
    for c in keep:
        c_idx = np.where(targets == c)[0]
        if per_class:
            c_idx = c_idx[:per_class]
        idx.extend(c_idx.tolist())
    return Subset(dataset, idx)


def make_synthetic_image_data(n, num_classes=4, size=224, seed=0):
    # 备用：4 类合成图像，用 3 个通道的亮度组合编码类别，保证可学习
    g = torch.Generator().manual_seed(seed)
    X = torch.rand(n, 3, size, size, generator=g) * 0.1
    y = torch.randint(0, num_classes, (n,), generator=g)
    for i in range(n):
        c = int(y[i])
        X[i, 0] += 0.9 if c in (0, 1) else 0.0
        X[i, 1] += 0.9 if c in (2, 3) else 0.0
        X[i, 2] += 0.9 if c in (0, 2) else (0.45 if c in (1, 3) else 0.0)
    return torch.utils.data.TensorDataset(X, y)


# 只保留 4 类：cat(3), dog(5), frog(6), ship(8)
KEEP = [3, 5, 6, 8]
remap = {3: 0, 5: 1, 6: 2, 8: 3}
CLASSES = ["cat", "dog", "frog", "ship"]

# CPU 上 ResNet18@224 很慢，自动缩小数据量与 epoch
PER_CLASS = 500 if device.type == "cuda" else 100
EPOCHS = 3 if device.type == "cuda" else 1
print(f"device={device.type} -> 每类 {PER_CLASS} 张, EPOCHS={EPOCHS}")

try:
    full_train = datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
    full_test = datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
    train_ds = RemapLabel(subset_by_classes(full_train, KEEP, PER_CLASS), remap)
    test_ds = RemapLabel(subset_by_classes(full_test, KEEP, max(PER_CLASS // 5, 20)), remap)
    print("使用 CIFAR-10 子集：train", len(train_ds), " test", len(test_ds))
except Exception as e:
    print("CIFAR-10 下载失败（可能网络较慢），改用合成图像数据继续演示：",
          type(e).__name__)
    train_ds = make_synthetic_image_data(4 * PER_CLASS, seed=0)
    test_ds = make_synthetic_image_data(200, seed=1)
    print("合成数据：train", len(train_ds), " test", len(test_ds))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)
print("batch 数:", len(train_loader), len(test_loader))

## 10.6 训练与评估函数（复用）

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = n = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            n += len(y)
    return correct / n


def train_model(model, train_loader, test_loader, epochs=5, lr=1e-3, freeze_backbone=True):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        if freeze_backbone:
            freeze_bn(model)             # 冻结主干时保持 BN 统计量
        total = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(y)
        acc = evaluate(model, test_loader)
        history.append(acc)
        print(f"  epoch {epoch}/{epochs} | loss {total/len(train_loader.dataset):.4f} | test acc {acc:.4f}")
    return history

### 训练：只训分类头

> 数据 resize 到 224 后，CPU 训练会比较慢。可用小 `epochs` 演示趋势。

In [ ]:
fe_model = get_resnet18(pretrained=True, num_classes=4).to(device)
# 冻结主干
for name, p in fe_model.named_parameters():
    if not name.startswith("fc"):
        p.requires_grad = False
fe_model.fc = nn.Linear(fe_model.fc.in_features, 4).to(device)

print("=== 特征提取：只训练分类头 ===")
hist_fe = train_model(fe_model, train_loader, test_loader, epochs=EPOCHS, lr=1e-3, freeze_backbone=True)
print("最终测试准确率:", hist_fe[-1])

## 10.7 策略二：微调（Fine-tune）

解冻最后若干层，用小学习率（`1e-4 ~ 1e-5`）训练。

分层学习率：底层小、顶层大——保住通用特征，又适应新任务。

In [ ]:
ft_model = get_resnet18(pretrained=True, num_classes=4).to(device)

# 先全部冻结
for p in ft_model.parameters():
    p.requires_grad = False

# 解冻 layer4 和 fc
for name, p in ft_model.named_parameters():
    if name.startswith("layer4") or name.startswith("fc"):
        p.requires_grad = True

ft_model.fc = nn.Linear(ft_model.fc.in_features, 4).to(device)

# 分层学习率
params = [
    {"params": [p for n, p in ft_model.named_parameters() if n.startswith("layer4")], "lr": 1e-4},
    {"params": [p for n, p in ft_model.named_parameters() if n.startswith("fc")], "lr": 1e-3},
]
optimizer = torch.optim.AdamW(params, weight_decay=1e-4)

print("可训练参数数:", sum(p.numel() for p in ft_model.parameters() if p.requires_grad))
print("layer4 lr=1e-4, fc lr=1e-3")

In [ ]:
print("=== 微调 layer4 + fc ===")
optimizer = torch.optim.AdamW(params, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
hist_ft = []
for epoch in range(1, EPOCHS + 1):
    ft_model.train()
    total = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(ft_model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    acc = evaluate(ft_model, test_loader)
    hist_ft.append(acc)
    print(f"  epoch {epoch}/{EPOCHS} | loss {total/len(train_loader.dataset):.4f} | test acc {acc:.4f}")
print("最终测试准确率:", hist_ft[-1])

In [ ]:
# 对比两种策略
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(hist_fe) + 1), hist_fe, "o-", label="特征提取（只训头部）")
plt.plot(range(1, len(hist_ft) + 1), hist_ft, "s-", label="微调 layer4+fc")
plt.xlabel("epoch"); plt.ylabel("测试准确率"); plt.legend(); plt.grid(alpha=0.3)
plt.title("迁移学习策略对比")
plt.show()

## 10.8 用 Hugging Face 迁移 NLP 模型（扩展）

CV 用 torchvision，NLP 现在主流是 Hugging Face `transformers`：

```python
# pip install transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)

enc = tokenizer(["I love this movie", "This is terrible"],
                padding=True, truncation=True, return_tensors="pt")
out = model(**enc).logits
```
思路完全一样：加载预训练 → 换头 → 微调（通常用 `Trainer` 或自己写循环）。

## 10.9 常见坑

| 坑 | 解决 |
|---|---|
| 冻结了层但 optimizer 仍传了全部参数 | 只传 `requires_grad=True` 的参数 |
| BN 统计量漂移 | 冻结时 `model.eval()` 对 BN，或设 `momentum=None` |
| 学习率太大破坏预训练权重 | 微调用小 lr（1e-5 ~ 1e-4） |
| 输入尺寸/归一化不匹配 | 必须用与预训练相同的 mean/std 和 resize |
| 训练慢 | 先缓存特征（特征提取时主干不回传，可预先算好） |

## 10.10 练习

**练习 1**：把 `KEEP` 改成 10 类全部，观察特征提取能到多少准确率。

**练习 2**：改 `per_class=100`（更少数据），比较从头训练 vs 迁移学习。

**练习 3**：解冻 `layer3 + layer4 + fc`，调整 lr 组合，看效果。

**练习 4**：用 `models.mobilenet_v2` 替换 ResNet18，比较参数量与准确率。

**练习 5**：把 `ResNet18` 换成 `ResNet50`，参数翻几倍？效果提升多少？

In [ ]:
# 练习 4：MobileNetV2
mb = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if True else None)
print("MobileNetV2 参数:", f"{sum(p.numel() for p in mb.parameters()):,}")
mb.classifier[1] = nn.Linear(mb.last_channel, 4)
print("换成 4 类后参数量:", f"{sum(p.numel() for p in mb.parameters()):,}")
print("ResNet18 参数:", f"{sum(p.numel() for p in get_resnet18(pretrained=False, num_classes=4).parameters()):,}")

## 10.11 小结

- 迁移学习 = 复用预训练特征；数据少时优先。
- **特征提取**：冻结主干（注意 BN），只训新头。
- **微调**：解冻部分层，小 lr，可用分层学习率。
- 输入预处理必须与预训练一致（resize、mean/std）。
- 现代实践：CV 用 torchvision，NLP 用 HuggingFace transformers。

下一节：`11_深度学习理论与优化技巧.ipynb`